## ライブラリインポート

In [ ]:
!pip install keras-core --upgrade
!pip install -q keras-nlp --upgrade

# This sample uses Keras Core, the multi-backend version of Keras.
# The selected backend is TensorFlow (other supported backends are 'jax' and 'torch')
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras
import keras_nlp
!pip install optuna
import optuna
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import re


##　データ読み込み

In [ ]:
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

### データ確認

In [ ]:
df_train.head(5)

In [ ]:
df_test.head(5)

In [ ]:
df_train["length"] = df_train["text"].apply(lambda x : len(x))
df_test["length"] = df_test["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(df_train["length"].describe())
print()

print("Test Length Stat")
print(df_test["length"].describe())

In [ ]:
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

EPOCHS = 2
AUTO = tf.data.experimental.AUTOTUNE

## 機械学習の準備

In [ ]:
from sklearn.model_selection import train_test_split

#目的変数と目的変数に分割
X = df_train["text"]
y = df_train["target"]

#評価用データを分割
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)

#テストデータ作成
X_test = df_test["text"]

In [ ]:
def objective(trial):
    tf.keras.backend.clear_session()
    
    # --- チューニングするハイパーパラメータの探索範囲を定義 ---
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    
    # --- モデルの構築 ---
    preset = "roberta_base_en"
    preprocessor = keras_nlp.models.RobertaPreprocessor.from_preset(preset, sequence_length=160)
    classifier = keras_nlp.models.RobertaClassifier.from_preset(
        preset,
        preprocessor=preprocessor,
        num_classes=2,
        dropout=dropout_rate
    )
    
    # --- モデルのコンパイルと学習 ---
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    classifier.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = classifier.fit(
        X_train, y_train,
        batch_size=32, # バッチサイズは固定
        epochs=5, 
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        verbose=0
    )
    
    return min(history.history['val_loss'])

# ===============================================================
# Step 3: Optunaによる最適化の実行
# ===============================================================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10) # 試行回数は環境に応じて調整してください

print("Best trial params: ", study.best_params)
print("Best val_loss: ", study.best_value)

# ===============================================================
# Step 4: 最適パラメータで最終モデルを学習
# ===============================================================
print("\nTraining final model with best hyperparameters...")

# Optunaで見つけた最適なパラメータを取得
best_params = study.best_params

# Kerasセッションをクリア
tf.keras.backend.clear_session()

# 最適なパラメータでモデルを再定義
preset = "roberta_base_en"
preprocessor = keras_nlp.models.RobertaPreprocessor.from_preset(preset, sequence_length=160)
final_classifier = keras_nlp.models.RobertaClassifier.from_preset(
    preset,
    preprocessor=preprocessor,
    num_classes=2,
    dropout=best_params['dropout_rate'] # 最適なドロップアウト率
)

optimizer = tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']) # 最適な学習率
final_classifier.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# "全て"の訓練データ(X, y)で学習
# EarlyStoppingは使わないため、エポック数は固定（探索で見つかった平均的なエポック数を参考にすると良い）
print("Fitting final model on all training data...")
final_classifier.fit(X, y, batch_size=32, epochs=4, verbose=1) # verbose=1で学習の進捗を表示


# ===============================================================
# Step 5: テストデータで予測し、提出ファイルを作成
# ===============================================================
print("\nPredicting on test data and creating submission file...")

X_test = df_test["text"]
predictions = final_classifier.predict(X_test)

# 提出用データフレームに予測結果を格納Mode spect
sample_submission["target"] = np.argmax(predictions, axis=1)

# CSVファイルとして出力
sample_submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' has been created successfully! 🎉")
print(sample_submission.head())